In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("MyApp").getOrCreate()

In [3]:
spark

Problem 1: Filter employees by salary

In [8]:
# Task: Return employees earning more than 50,000, displaying only name and salary.
from pyspark.sql import functions as F

data = [
    (1, "Ravi", 40000),
    (2, "Priya", 65000),
    (3, "Arun", 55000),
    (4, "Kiran", 30000)
]

columns = ["emp_id", "name", "salary"]

df = spark.createDataFrame(data, columns)

df.filter(df.salary>50000).show()



+------+-----+------+
|emp_id| name|salary|
+------+-----+------+
|     2|Priya| 65000|
|     3| Arun| 55000|
+------+-----+------+



<b>Problem 2: Calculate employee bonuses</b>  
<font color="green"> Task: Add a bonus column with 10% of salary for employees earning more than 50,000, and 5% for everyone else.</font>

In [4]:

from pyspark.sql import functions as F
data = [
    ("Ravi", 40000),
    ("Priya", 65000),
    ("Arun", 55000)
]

columns = ["name", "salary"]

df = spark.createDataFrame(data, columns)

display(df)

df.withColumn("bouns",F.when(df.salary > 50000,(df.salary)*0.10).otherwise(df.salary*0.05)).show()
# Your task:
# Salary > 50000: bonus = 10%
# Otherwise: bonus = 5%
# Create a new column called bonus


DataFrame[name: string, salary: bigint]

+-----+------+------+
| name|salary| bouns|
+-----+------+------+
| Ravi| 40000|2000.0|
|Priya| 65000|6500.0|
| Arun| 55000|5500.0|
+-----+------+------+



<b>Problem 3: Department-wise salary totals</b>  
Task: Group employees by department and calculate the total salary. Use the DataFrame API, not Spark SQL.

In [11]:

from pyspark.sql import functions as F

data = [
    ("Ravi", "IT", 40000),
    ("Priya", "HR", 65000),
    ("Arun", "IT", 55000),
    ("Kiran", "HR", 30000),
    ("Sneha", "IT", 60000)
]

columns = ["name", "department", "salary"]

df = spark.createDataFrame(data, columns)

display(df)

# Your task:
# Group by department
# Calculate total salary
# Rename the result column to total_salary

df.groupBy("department").agg(F.sum("salary").alias("total_salary")).show()

DataFrame[name: string, department: string, salary: bigint]

+----------+------------+
|department|total_salary|
+----------+------------+
|        HR|       95000|
|        IT|      155000|
+----------+------------+



<b>Problem4: Pipeline status classification</b>

In [4]:

from pyspark.sql import functions as F

data = [
    ("SAP_A6P", 15),
    ("SAP_F6P", 45),
    ("SAP_H2P", 30),
    ("SAP_N6P", 60)
]

columns = ["pipeline", "duration_min"]

df = spark.createDataFrame(data, columns)

display(df)

# Your task:
# duration_min > 30  -> SLOW
# duration_min <= 30 -> NORMAL
# Create a status column

df.withColumn("status",F.when(df.duration_min>30,"SLOW").otherwise("NORMAL")).show()


DataFrame[pipeline: string, duration_min: bigint]

+--------+------------+------+
|pipeline|duration_min|status|
+--------+------------+------+
| SAP_A6P|          15|NORMAL|
| SAP_F6P|          45|  SLOW|
| SAP_H2P|          30|NORMAL|
| SAP_N6P|          60|  SLOW|
+--------+------------+------+



In [7]:

from pyspark.sql import functions as F

bronze_data = [
    ("C101", "Bengaluru", "Retail", "2026-11-08", "2026-11-09 09:00:00"),
    ("C101", "Pune", "Retail", "2026-11-03", "2026-11-10 10:00:00"),
    ("C101", "Bengaluru", "Retail", "2026-11-08", "2026-11-10 11:00:00"),
    ("C102", "Mumbai", "Enterprise", "2026-11-07", "2026-11-09 08:00:00"),
    ("C103", "Delhi", "Retail", "2026-11-06", "2026-11-09 07:00:00")
]

bronze_df = spark.createDataFrame(
    bronze_data,
    ["customer_id", "city", "segment",
     "effective_date", "ingestion_time"]
).withColumn(
    "effective_date", F.to_date("effective_date")
).withColumn(
    "ingestion_time", F.to_timestamp("ingestion_time")
)

silver_data = [
    ("C101", "Hyderabad", "Retail", "2026-11-01", "2026-11-05", False),
    ("C101", "Chennai", "Retail", "2026-11-05", None, True),
    ("C102", "Mumbai", "Wholesale", "2026-11-02", None, True)
]

silver_schema = """
customer_id STRING,
city STRING,
segment STRING,
valid_from STRING,
valid_to STRING,
is_current BOOLEAN
"""

silver_df = spark.createDataFrame(
    silver_data, silver_schema
).withColumn(
    "valid_from", F.to_date("valid_from")
).withColumn(
    "valid_to", F.to_date("valid_to")
)

display(bronze_df)
display(silver_df)

# Your task:
# Deduplicate incoming changes
# Handle historical updates
# Maintain accurate validity intervals
bronze_df.show()
silver_df.show()


DataFrame[customer_id: string, city: string, segment: string, effective_date: date, ingestion_time: timestamp]

DataFrame[customer_id: string, city: string, segment: string, valid_from: date, valid_to: date, is_current: boolean]

+-----------+---------+----------+--------------+-------------------+
|customer_id|     city|   segment|effective_date|     ingestion_time|
+-----------+---------+----------+--------------+-------------------+
|       C101|Bengaluru|    Retail|    2026-11-08|2026-11-09 09:00:00|
|       C101|     Pune|    Retail|    2026-11-03|2026-11-10 10:00:00|
|       C101|Bengaluru|    Retail|    2026-11-08|2026-11-10 11:00:00|
|       C102|   Mumbai|Enterprise|    2026-11-07|2026-11-09 08:00:00|
|       C103|    Delhi|    Retail|    2026-11-06|2026-11-09 07:00:00|
+-----------+---------+----------+--------------+-------------------+

+-----------+---------+---------+----------+----------+----------+
|customer_id|     city|  segment|valid_from|  valid_to|is_current|
+-----------+---------+---------+----------+----------+----------+
|       C101|Hyderabad|   Retail|2026-11-01|2026-11-05|     false|
|       C101|  Chennai|   Retail|2026-11-05|      NULL|      true|
|       C102|   Mumbai|Wholesale|2

<b>Problem5: Count pipeline failures by SAP system</b>

In [9]:

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType
)

data = [
    ("A6P", "silver_stream", "FAILED"),
    ("F6P", "bronze_load", "SUCCESS"),
    ("A6P", "bronze_load", "FAILED"),
    ("H2P", "silver_stream", "FAILED"),
    ("F6P", "silver_stream", "FAILED"),
    ("A6P", "silver_stream", "SUCCESS"),
    ("F6P", "bronze_load", "FAILED")
]

schema = StructType([
    StructField("system", StringType(), False),
    StructField("job_name", StringType(), False),
    StructField("status", StringType(), False)
])

df = spark.createDataFrame(data, schema)

display(df)

# Your solution starts below.
# 1. Keep only FAILED records.
# 2. Group records by system.
# 3. Count failures.
# 4. Name the output column failure_count.

df.filter(df.status=="FAILED").groupBy(df.system).agg(F.count(df.job_name).alias("Failuer_count")).show()


DataFrame[system: string, job_name: string, status: string]

+------+-------------+
|system|Failuer_count|
+------+-------------+
|   A6P|            2|
|   H2P|            1|
|   F6P|            2|
+------+-------------+

